# 03 · Verifier Agent: crop-and-zoom verification + crop-margin tuning

**Purpose.** Tune the crop expansion margin on held-out, then run the frozen VLM verifier on every cached candidate of held-out and test. Also runs the global-view ablation (verifier sees the full image) and, optionally, the fully label-blind open-answer mode.

| | |
|---|---|
| **Proposal objective** | Objective 1 (Verifier Agent), Objective 3 (ablation of evidence separation) |
| **Inputs** | `results/detections/*.jsonl` |
| **Outputs** | `results/verifier/*.jsonl`, `results/tuning/selected.json` (crop_margin), `results/verifier/*.csv` |
| **Runtime (T4)** | fast profile: margin search ≈5 min; held-out + test ≈10–15 min; global ablation ≈5–10 min |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess, shutil  # paths, sys.path, shell commands (git clone, pip), folder copy
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = (not IN_COLAB) and os.path.isdir("/kaggle/working")    # Kaggle notebook (needs Internet ON, see docs/KAGGLE_GUIDE.md)

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"
BRANCH = "kaggle-version"  # git branch to take the latest code from (change it if you push to another branch)

if IN_COLAB:
    from google.colab import drive  # Colab only: mount Google Drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, DRIVE_PROJECT], check=True)
    elif os.path.isdir(os.path.join(DRIVE_PROJECT, ".git")):   # pick up the latest pushed code (never touches your notebooks/results)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "fetch", "origin", BRANCH], check=False)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "checkout", "FETCH_HEAD", "--", "src", "configs", "requirements.txt", "tools", "app"], check=False)
    PROJECT_ROOT = DRIVE_PROJECT
elif IN_KAGGLE:
    PROJECT_ROOT = "/kaggle/working/detect-verify-decide"
    url = REPO_URL
    if REPO_URL:
        try:   # private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons -> Secrets)
            from kaggle_secrets import UserSecretsClient
            url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
        except Exception:
            pass
    if not os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")):
        import glob
        attached = glob.glob("/kaggle/input/*/src/dvd") + glob.glob("/kaggle/input/*/*/src/dvd")   # project attached as a Kaggle Dataset
        if attached:
            shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), PROJECT_ROOT)
        elif REPO_URL:
            r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT_ROOT], capture_output=True)
            if r.returncode:   # (the message deliberately does not echo the command, which could contain the token)
                raise RuntimeError("git clone failed: is Internet ON, REPO_URL / BRANCH right, and for a private repo the GITHUB_TOKEN secret set?")
            subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", REPO_URL])   # never keep the token in .git/config
        else:
            raise RuntimeError("Set REPO_URL (and BRANCH) above, or attach the project as a Kaggle Dataset.")
    elif REPO_URL and os.path.isdir(os.path.join(PROJECT_ROOT, ".git")):   # folder left over from an earlier run: update it to the latest code
        subprocess.run(["git", "-C", PROJECT_ROOT, "fetch", "-q", url, BRANCH], capture_output=True)
        subprocess.run(["git", "-C", PROJECT_ROOT, "checkout", "-q", "FETCH_HEAD", "--", "src", "configs", "requirements.txt", "tools", "app"], capture_output=True)
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab / Kaggle already have torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB or IN_KAGGLE:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt  # json files, numpy arrays, pandas tables, matplotlib plots
from dvd.config import load_config, save_selected, results_dir  # load configs/default.yaml, save tuned values, results folders
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache  # seeding, JSON helpers and the resumable JSONL cache
from dvd import paths  # cache-file path helpers

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch  # PyTorch: GPU check and memory statistics
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.data.coco import prepare_annotations, fetch_images, CocoData, load_splits  # COCO annotations, image download, splits
prepare_annotations(cfg)   # annotation json only (the 241 MB zip is cached on Drive)
coco = CocoData(cfg)
splits = load_splits(cfg)
HELDOUT, TEST = splits["heldout"], splits["test"]
fetch_images(cfg, HELDOUT + TEST)   # only these images (a few hundred files), not the whole 778 MB val2017 zip
print(f"held-out: {len(HELDOUT)} images | test: {len(TEST)} images")

In [ ]:
from dvd.agents.verifier import build_verifier  # Verifier Agent (frozen Qwen2-VL): factory and crop builder
from dvd.pipeline import run_verification_stage  # cached offline stages and the online Detect-Verify-Decide pipeline
from dvd.experiments import SplitData  # methods over identical candidates: sweeps, operating points, evaluation, costs
from dvd.metrics import auroc, TP, FP  # matching, metrics, sweeps, AUROC, POPE scores
H = SplitData(cfg, coco, "heldout", HELDOUT)
T = SplitData(cfg, coco, "test", TEST)
verifier = build_verifier(cfg)

## 1. Crop-margin search (held-out subset)
Criterion: AUROC of the verifier's P(yes) for separating true from false candidates. Also reported: how many FPs it would reject and how many TPs it would wrongly reject at P(yes)=0.5.

In [ ]:
tune_ids = HELDOUT[: cfg["margin_tuning"]["images"]]
rows = []
for m in cfg["margin_tuning"]["margins"]:
    ver = run_verification_stage(verifier, H.det, tune_ids, paths.ver_cache(cfg, "heldout", "crop", margin=m), cfg,
                                 coco.image, view="crop", margin=m, desc=f"margin {m}")
    y = np.concatenate([H.labels[i] for i in tune_ids]); p = np.concatenate([np.asarray(ver[i]["p_yes"]) for i in tune_ids])
    keep = y != -1; y, p = y[keep], p[keep]
    rows.append({"margin": m, "auroc": auroc(p, y == TP), "fp_rejected": float((p[y == FP] <= .5).mean()),
                 "tp_over_rejected": float((p[y == TP] <= .5).mean()), "yes_rate": float((p > .5).mean()),
                 "n_candidates": int(len(y))})
margin_df = pd.DataFrame(rows); margin_df.to_csv(results_dir(cfg, "verifier") / "margin_tuning.csv", index=False)
margin_df

In [ ]:
best_margin = float(margin_df.loc[margin_df.auroc.idxmax(), "margin"])
save_selected(cfg, crop_margin=best_margin)
print("selected crop margin:", best_margin)
margin_df.plot(x="margin", y=["auroc", "fp_rejected", "tp_over_rejected"], marker="o", grid=True, title="Crop margin search"); plt.show()

## 2. Crop verification on all candidates (proposed verifier)

In [ ]:
ver_h = run_verification_stage(verifier, H.det, HELDOUT, paths.ver_cache(cfg, "heldout", "crop"), cfg, coco.image, view="crop", desc="verify held-out")
ver_t = run_verification_stage(verifier, T.det, TEST, paths.ver_cache(cfg, "test", "crop"), cfg, coco.image, view="crop", desc="verify test")

## 3. Global-view ablation
Same verifier, same question, but it sees the **whole image** instead of the crop. Isolates the contribution of evidence separation (crop-and-zoom).

In [ ]:
RUN_GLOBAL_ABLATION = True
if RUN_GLOBAL_ABLATION:
    gl_h = run_verification_stage(verifier, H.det, HELDOUT, paths.ver_cache(cfg, "heldout", "full"), cfg, coco.image, view="full", desc="global held-out")
    gl_t = run_verification_stage(verifier, T.det, TEST, paths.ver_cache(cfg, "test", "full"), cfg, coco.image, view="full", desc="global test")

## 4. Verifier diagnostics on test

In [ ]:
def diag(ver, data, name):
    # Verifier diagnostics on a split: AUROC, share of FPs rejected, share of TPs wrongly rejected, yes-rate, time per call.
    y = np.concatenate([data.labels[i] for i in data.ids]); p = np.concatenate([np.asarray(ver[i]["p_yes"]) for i in data.ids])
    k = y != -1; y, p = y[k], p[k]
    t = sum(r["verify_time_s"] for r in ver.values()) / max(1, sum(r["verify_invocations"] for r in ver.values()))
    return {"view": name, "auroc": auroc(p, y == TP), "fp_rejected@0.5": float((p[y == FP] <= .5).mean()),
            "tp_over_rejected@0.5": float((p[y == TP] <= .5).mean()), "yes_rate": float((p > .5).mean()), "s_per_call": t}, (y, p)
d_crop, (y, p_crop) = diag(ver_t, T, "crop")
rows = [d_crop]
if RUN_GLOBAL_ABLATION:
    d_full, (_, p_full) = diag(gl_t, T, "full image"); rows.append(d_full)
diag_df = pd.DataFrame(rows); diag_df.to_csv(results_dir(cfg, "verifier") / "verifier_diagnostics_test.csv", index=False)
diag_df

In [ ]:
fig, axes = plt.subplots(1, 2 if RUN_GLOBAL_ABLATION else 1, figsize=(12, 4), squeeze=False)
for ax, (p, name) in zip(axes[0], [(p_crop, "crop"), (p_full if RUN_GLOBAL_ABLATION else None, "full image")]):
    ax.hist(p[y == TP], bins=40, alpha=.6, label="true detections", density=True)
    ax.hist(p[y == FP], bins=40, alpha=.6, label="false positives", density=True)
    ax.set(title=f"Verifier P(yes) - {name} view", xlabel="P(yes)"); ax.legend()
plt.show()

## 5. (Optional) fully label-blind open-answer mode
The VLM is asked *what* the object is, without being told the label; the answer is matched to the label afterwards. Slower (generation instead of one forward pass) — run on the margin-tuning subset only.

In [ ]:
RUN_OPEN_MODE = False
if RUN_OPEN_MODE:
    op = run_verification_stage(verifier, H.det, tune_ids, paths.ver_cache(cfg, "heldout", "crop", mode="open"), cfg,
                                coco.image, view="crop", mode="open", desc="open mode")
    y = np.concatenate([H.labels[i] for i in tune_ids]); v = np.concatenate([np.asarray(op[i]["verdict"], bool) for i in tune_ids])
    k = y != -1
    print(f"open mode: FP rejected {(~v[k][y[k] == FP]).mean():.3f} | TP over-rejected {(~v[k][y[k] == TP]).mean():.3f}")
    print("sample answers:", [a for i in tune_ids[:5] for a in (op[i]["answers"] or [])][:15])

**Next:** `04_verifier_pope_eval` (verifier in isolation) and `05_decision_agent_and_pipeline`.